# 第3回A 演習: 言語モデルの実装 (ブラウザ版)

スライド: [第3回A ブラウザ版](../03a-llm-webllm/) / Colab 版は[こちら](https://colab.research.google.com/gist/3tty0n/1c0f3cf4d3f4fb2f30354677156e75ea/03a_llm.ipynb)

Qwen2.5-0.5B をブラウザの中 (WebGPU) で動かします。コードはすべて Python です。

- **テンソル演算** (softmax / topk / 温度 / top-p) → `torch` (torch-pyodide)
- **モデルの生成** (LLM 本体) → WebLLM (Pyodide が内部でブリッジ)

- **WebGPU 対応ブラウザ** (Chrome / Edge の最新版) で開くこと
- 初回だけモデルと torch ランタイムをダウンロード (約 0.5 GB、Wi-Fi 推奨)。2回目からキャッシュ
- 生成中はタブを開いたままにする
- このノートブックに `score()` は無い。各課題の**出力を残したまま**保存して提出

**torch について**: torch-pyodide は本家 PyTorch のブラウザ移植 (実験版) です。
transformers や学習済みモデルの重みは含まないため、LLM 本体は WebLLM を使います。
Colab 版との違いは、`torch.topk`/`torch.sort` がタプルを返すことと、
テンソルを添え字にできない (`.item()` でスカラー化) ことです。

---

## セットアップ

下のセルを1回実行する。初回はダウンロードに数分かかる。

In [ ]:
# [SKIP] 採点の対象外
# ---- セットアップ (最初に1回だけ実行) ----
# WebGPU 対応ブラウザ (Chrome / Edge の最新版) で実行すること
import js
from js import Object
from pyodide.ffi import to_js

# torch 本体 (torch-pyodide) を読み込む
_manifest = await js.Function(
    'u', 'return (async()=>{const r=await fetch(u); return await r.json()})(u)')(
    'https://celsowm.github.io/torch-pyodide/latest.json')
_runtime = await js.Function('u', 'return import(u)')(_manifest.runtimeUrl)
_runtime.installTorchRuntime(js.globalThis)

import micropip
await micropip.install(str(_manifest.wheelUrl))
import torch
print("torch:", torch.__version__, "| GPU:", torch.cuda.is_available())

# LLM (WebLLM) を読み込む
webllm = await js.Function('u', 'return import(u)')(
    'https://esm.run/@mlc-ai/web-llm@0.2.84',
)
MODEL = "Qwen2.5-0.5B-Instruct-q4f16_1-MLC"
print("キャッシュ済み:", bool(await webllm.hasModelInCache(MODEL)))

def _progress(r):
    print(f"[読み込み中 {r.progress:>4.0%}] {r.text}")

engine = await webllm.CreateMLCEngine(
    MODEL,
    to_js({"initProgressCallback": _progress}, dict_converter=Object.fromEntries),
)
print("準備完了:", MODEL)

In [ ]:
# [SKIP] 採点の対象外
# ---- ヘルパー (Colab 版と同じ torch の書き方) ----

def sample_next(logits, temperature=0.8, k=50):
    # 温度で割って softmax → 上位 k 個に絞ってから重み付きで選ぶ
    probs = torch.softmax(logits / temperature, dim=-1)
    values, indices = torch.topk(probs, k)
    return indices[torch.multinomial(values, 1).item()].item()

def sample_next_top_p(logits, temperature=0.8, p=0.9):
    # 確率の高い順に累積し、合計が p に達した候補だけ残す
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, descending=True)
    cumsum = torch.cumsum(sorted_probs)
    mask = cumsum - sorted_probs < p
    return sorted_idx[mask][torch.multinomial(sorted_probs[mask], 1).item()].item()

def to_payload(messages, **opts):
    # Python の辞書を JavaScript のオブジェクトに変換する
    return to_js({"messages": messages, **opts},
                 dict_converter=Object.fromEntries)

async def ask(prompt, **opts):
    # 1往復の応答を文字列で返す
    opts.setdefault("max_tokens", 32)
    reply = await engine.chat.completions.create(
        to_payload([{"role": "user", "content": prompt}], **opts))
    return reply.choices[0].message.content.strip()

In [ ]:
# [SKIP] 採点の対象外
# ---- 動作確認 ----
print("応答:", await ask("日本の首都は？"))

# 温度の効果 (Colab 版と同じ結果になる)
logits = torch.tensor([3.0, 1.0])
print(torch.softmax(logits / 0.1, dim=-1))  # 低温は決め打ちに近い
print(torch.softmax(logits / 2.0, dim=-1))  # 高温はばらける

---

## 課題1: トークンを数える

`reply.usage` に入力と出力のトークン数が入っている。
`prompt_tokens` と `completion_tokens` を表示しよう。

```
出力例:
入力のトークン数: 7
作ったトークン数: 16
```

In [ ]:
# 課題1: usage で、入力と出力それぞれのトークン数を表示しよう
reply = await engine.chat.completions.create(to_payload(
    [{"role": "user", "content": "むかしむかし、あるところに"}],
    max_tokens=16,
))

print("続き:", reply.choices[0].message.content.strip())

# TODO: reply.usage.prompt_tokens と reply.usage.completion_tokens を表示

---

## 課題2: greedy の確認

`temperature: 0` でランダム性が消え、毎回同じ出力になる。
同じ入力を2回投げて、結果が一致することを確かめよう。

In [ ]:
# 課題2: temperature 0 で、同じ入力から同じ出力が返ることを確かめよう
async def ask_greedy(prompt):
    reply = await engine.chat.completions.create(to_payload(
        [{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=24,
    ))
    return reply.choices[0].message.content.strip()

a = await ask_greedy("好きな食べ物は")
b = await ask_greedy("好きな食べ物は")

# TODO: a と b を表示し、a == b が True になることを確認

---

## 課題3: 温度と top-k

確率分布の形は温度で変わる。ダミーの logits に `sample_next` を
適用し、`0.3 / 0.8 / 1.5` の3つで選ばれる候補を比べよう。

```
出力例 (同じ乱数でも、温度によって選ばれ方が変わる):
T=0.3: ... (強い候補ばかり)
T=1.5: ... (低い候補も混ざる)
```

In [ ]:
# 課題3: 温度 0.3 / 0.8 / 1.5 で選ばれる候補を比べよう
logits = torch.randn(100)  # ダミーの logits (語彙サイズ 100)

# TODO: 3つの温度で sample_next を呼び、何度か試して結果を比べる
for t in [0.3, 0.8, 1.5]:
    picked = [int(sample_next(logits, temperature=t, k=10)) for _ in range(8)]
    print(f"T={t}: {picked}")

**まとめ** (書き換えてよい): 低温は強い候補ばかり、高温は低い候補も混ざる。

---

## 課題4: 履歴による会話

モデルは履歴を覚えない。過去のやりとりを毎回 `messages` に全部入れて
渡すのが呼び出し側の仕事。3往復の会話を完成させよう。

In [ ]:
# 課題4: 3往復の会話を作ろう
history = [
    {"role": "system", "content": "あなたは簡潔に答えるアシスタントです。"},
]
questions = ["日本の首都は？", "そこの人口は？", "有名な食べ物は？"]

for q in questions:
    history.append({"role": "user", "content": q})
    # TODO: engine.chat.completions.create に history を渡し、
    #       応答を text に受け取る (max_tokens=48)
    text = ...
    # TODO: history に assistant の応答を追加する
    print(f"Q: {q}")
    print(f"A: {text}\n")

---

## 発展: top-p と繰り返しペナルティ

top-p は torch の `sample_next_top_p` で、繰り返しペナルティは
WebLLM の `presence_penalty` で試せる。

In [ ]:
# 発展1: top-p (ダミーの logits で候補の広がりを見る)
logits = torch.randn(100)
print([int(sample_next_top_p(logits, temperature=0.8, p=0.9)) for _ in range(8)])
print([int(sample_next_top_p(logits, temperature=0.8, p=0.5)) for _ in range(8)])

In [ ]:
# 発展2: 繰り返しペナルティ (実際のモデル)
reply = await engine.chat.completions.create(to_payload(
    [{"role": "user", "content": "好きな食べ物は"}],
    temperature=0.8,
    presence_penalty=0.5,
    max_tokens=48,
))
print(reply.choices[0].message.content.strip())

---

## 提出

- すべてのセルを上から実行し、**出力を残したまま** `File → Download` で .ipynb を保存
- 提出は kibaco から。採点ページ (grade.html) はこの回には使えない
- 詰まったらスライドの[課題一覧](../03a-llm-webllm/)に戻る
- Colab が使える人は Colab 版 (torch) でも構わない。課題はどちらも同じ